In [ ]:
# Загрузка файлов и подготовка окружения
!gdown "13ufoiykXawSNVJCpRnEn80eL6e29WEuD"  # https://drive.google.com/file/d/13ufoiykXawSNVJCpRnEn80eL6e29WEuD/view?usp=sharing
!unzip -q smpl.zip -d /content/
!rm smpl.zip
!pip install /content/wheels/pytorch3d*.whl /content/wheels/ultralytics*.whl

# SMPL: Skinned Multi-Person Linear Model

## Содержание:

**Часть 1. Вспомнить теорию SMPL**
1. Откуда взялся SMPL
2. Параметры формы ($\beta$) и позы ($\theta$)
3. Linear Blend Skinning (LBS) и кинематическое дерево
4. Регрессор суставов

**Часть 2. Пайплайн восстановления SMPL по мультивью-изображениям**
1. Обзор пайплайна
2. Детекция человека через YOLO
3. Оценка 2D-позы через ViTPose
4. Триангуляция скелета в 3D
5. Оптимизация SMPL-параметров к ViTPose-скелету
6. Уточнение параметров SMPL по маскам
7. `✨Бонус` Трекинг по видео

In [ ]:
%matplotlib inline

import sys
import os
import json
import warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Markdown

# Убиваем мешающие варнинги NNPACK от PyTorch (CPU без NNPACK-ускорения)
warnings.filterwarnings("ignore", message=".*NNPACK.*")
os.environ["PYTHONWARNINGS"] = "ignore::UserWarning"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# Рабочие пути
ROOT = Path(".").resolve()
DATA_DIR = ROOT / "data"
MODELS_DIR = ROOT / "models"
SMPL_MODEL_PATH = MODELS_DIR / "SMPL_NEUTRAL.pkl"

print(f"Root: {ROOT}")
print(f"SMPL model: {SMPL_MODEL_PATH} (exists: {SMPL_MODEL_PATH.exists()})")

# Часть 1. Вспомнить теорию SMPL

## 1.1 Откуда взялся SMPL

### Зачем нужна параметрическая модель тела?

В задачах 3D-зрения часто нужно восстанавливать форму и позу человека. Можно работать с произвольными мешами, но это сложно — слишком много степеней свободы. Параметрическая модель позволяет описать тело человека **компактным набором параметров**.

### Эволюция моделей тела

| Год  | Модель     | Описание |
|------|------------|-------------|
| 2005 | **SCAPE**  | Первая широко известная модель, объединяющая зависимость деформаций от формы и позы; использовала деформации треугольников (triangle deformations) и была вычислительно тяжелой/медленной. |
| 2015 | **SMPL**   | Переход на Linear Blend Skinning (LBS) + pose/shape blend shapes; быстрая и дифференцируемая, удобно встраивается в оптимизацию и learning-пайплайны. |
| 2019 | **SMPL-X** | SMPL, расширенная до экспрессивной модели с детальными руками (MANO) и лицом (FLAME), чтобы одним набором параметров описывать тело+кисти+мимику. |
| 2020 | **STAR**   | Sparse-версия SMPL: делает позозависимые деформации локальными (избегая «дальних» корреляций, когда движение в одной части тела влияет на далекую), при этом заметно сокращает число параметров и учитывает зависимость деформаций от телосложения (BMI). |
| 2022 | **SUPR**   | Sparse unified part-based представление: акцент на разреженных (локальных) деформациях и «модульности» по частям тела, чтобы лучше контролировать, какие области меша затрагивает конкретная артикуляция/часть. |
| 2025 | **SKEL**   | Модель, ориентированная на биомеханически более корректный внутренний скелет: делает шаг от чисто skin-моделей к реконструкции/использованию анатомически правдоподобного скелета и кинематики. |


**SMPL — ключевые идеи:**
1. Обучена на ~4000 3D-сканов реальных людей (базы CAESAR, и др.).
2. Разделяет **форму** (shape) и **позу** (pose).
3. Использует **Linear Blend Skinning** для финальной деформации — быстро и дифференцируемо.
4. Имеет **6890 вершин**, **23 сустава + 1 корень**.
5. Управляется **82 параметрами**: 10 на форму ($\beta$) и 72 на позу ($\theta$).
$$M(\beta, \theta) : \mathbb{R}^{|\beta| + |\theta|} \rightarrow \mathbb{R}^{6890 \times 3}$$

## 1.2 Параметры модели: $\beta$ и $\theta$

### Shape параметры $\beta \in \mathbb{R}^{10}$

Параметры формы $\beta$ получены через **PCA** (Principal Component Analysis) по базе 3D-сканов:

1. Берём ~4000 сканов людей, регистрируем их на общий шаблон (6890 вершин)
2. Получаем матрицу данных: каждая строка — 6890×3 = 20670 координат одного человека
3. Вычитаем среднее (template) $\bar{T}$ и делаем PCA
4. Первые 10 компонент = первые 10 «направлений» вариации формы тела

$$T_s(\beta) = \bar{T} + \sum_{i=1}^{10} \beta_i \cdot S_i$$

где $S_i \in \mathbb{R}^{6890 \times 3}$ — shape blend shape (PCA-компонента).

**Что кодируют $\beta$:** рост, полноту, пропорции тела, ширину плеч и т.д.

### Pose параметры $\theta \in \mathbb{R}^{72}$

Поза описывается через **axis-angle** повороты 24 суставов:

$$\theta = [\theta_0, \theta_1, \ldots, \theta_{23}], \quad \theta_i \in \mathbb{R}^3$$

- $\theta_0$ — глобальная ориентация (pelvis)
- $\theta_1, \ldots, \theta_{23}$ — локальные повороты остальных суставов
- Axis-angle: вектор $\omega$, направление = ось поворота, длина $\|\omega\|$ = угол
- Преобразование в матрицу поворота осуществляется по **формула Родригеса**

### Визуализация: влияние $\beta$ на форму

In [ ]:
from utils.smpl_model import SMPL, SMPL_SKELETON, SMPL_JOINT_NAMES
from utils.visualization import plot_mesh_3d

# Загрузка модели на DEVICE
smpl = SMPL(str(SMPL_MODEL_PATH), device=DEVICE)
print(f"SMPL model loaded on {DEVICE}: {smpl.n_verts} vertices, {smpl.n_joints} joints")
print(f"Faces: {smpl.faces.shape}")

In [ ]:
import plotly.graph_objects as go
from utils.visualization import plot_meshes_comparison

theta_zero = torch.zeros(1, 72, device=DEVICE)
faces_np = smpl.faces.cpu().numpy()

# beta_0 отвечает преимущественно за рост
beta_vals = [-3.0, 0.0, 3.0]
meshes, labels = [], []
for bv in beta_vals:
    beta = torch.zeros(1, 10, device=DEVICE)
    beta[0, 0] = bv
    verts, _ = smpl(beta, theta_zero)
    meshes.append(verts[0])
    labels.append(f"\u03b2\u2080={bv:+.0f}")

fig = plot_meshes_comparison(
    meshes, faces_np, labels=labels,
    title="\u03b2\u2080: влияние на рост",
    mode="side_by_side", spacing=2.0, opacity=0.5,
)
fig.show()

In [ ]:
# beta_1 преимущественно влияет на полноту
beta_vals = [-3.0, 0.0, 3.0]
meshes, labels = [], []
for bv in beta_vals:
    beta = torch.zeros(1, 10, device=DEVICE)
    beta[0, 1] = bv
    verts, _ = smpl(beta, theta_zero)
    meshes.append(verts[0])
    labels.append(f"\u03b2\u2081={bv:+.0f}")

fig = plot_meshes_comparison(
    meshes, faces_np, labels=labels,
    title="\u03b2\u2081: влияние на полноту",
    mode="side_by_side", spacing=2.0, opacity=0.5,
)
fig.show()

### Визуализация: влияние $\theta$ на позу

In [ ]:
# Влияние theta: overlay на одной сцене
beta_zero = torch.zeros(1, 10, device=DEVICE)

pose_defs = [
    ("T-поза", torch.zeros(1, 72, device=DEVICE)),
    ("Левая рука", None),
    ("Колени согнуты", None),
]

# Поднимаем левую руку (joint 16 = left_shoulder)
theta_arm = torch.zeros(1, 72, device=DEVICE)
theta_arm[0, 16*3 + 2] = -1.2
pose_defs[1] = ("Левая рука", theta_arm)

# Сгибаем колени (joints 4, 5)
theta_knees = torch.zeros(1, 72, device=DEVICE)
theta_knees[0, 4*3] = 0.8
theta_knees[0, 5*3] = 0.8
pose_defs[2] = ("Колени согнуты", theta_knees)

meshes, labels = [], []
for label, theta in pose_defs:
    verts, _ = smpl(beta_zero, theta)
    meshes.append(verts[0])
    labels.append(label)

fig = plot_meshes_comparison(
    meshes, faces_np, labels=labels,
    title="\u03b8: разные позы",
    mode="overlay", opacity=0.35,
)
fig.show()

## 1.3 Linear Blend Skinning (LBS)

### Кинематическое дерево

SMPL задаёт **иерархическую структуру** (kinematic tree) из 24 суставов:

![tree](assets/smpl_tree.jpg)

### Forward Kinematics (FK)

Глобальное преобразование сустава $k$ — это **произведение всех локальных поворотов** от корня до сустава $k$:

$$G_k = G_{\text{parent}(k)} \cdot \begin{bmatrix} R_k & J_k - J_{\text{parent}(k)} \\ 0 & 1 \end{bmatrix}$$

Для корня (pelvis): $G_0 = \begin{bmatrix} R_0 & J_0 \\ 0 & 1 \end{bmatrix}$

### LBS формула

Финальная позиция каждой вершины — **взвешенная сумма** преобразований от всех суставов:

$$v'_i = \left( \sum_{k=1}^{K} w_{ik} \cdot G'_k \right) \cdot \begin{bmatrix} v_i \\ 1 \end{bmatrix}$$

где:
- $w_{ik}$ — вес влияния сустава $k$ на вершину $i$ ($\sum_k w_{ik} = 1$)
- $G'_k = G_k \cdot \begin{bmatrix} I & -J_k \\ 0 & 1 \end{bmatrix}$ — преобразование с вычтенной rest-позой

Говоря иными словами, вершина привязана к нескольким ближайшим суставам. И при вращении сустава вершина двигается пропорционально своему весу привязки.

### Интерактив: 2D кинематическая цепь

In [ ]:
from utils.lbs_demo import create_lbs_chain_demo

create_lbs_chain_demo(n_joints=5, segment_length=1.0)

### Интерактив: LBS-веса и смешивание

In [ ]:
from utils.lbs_demo import create_lbs_weights_demo

create_lbs_weights_demo()

## 1.4 Регрессор суставов (Joint Regressor)

### Как задаются суставы?

В SMPL суставы **не заданы явно** - они вычисляются из вершин меша:

$$J = \mathscr{J} \cdot T_s(\beta)$$

где $\mathscr{J} \in \mathbb{R}^{24 \times 6890}$ — **матрица-регрессор** суставов.

Каждая строка $\mathscr{J}_k$ — это набор весов, показывающих, какие вершины "определяют" положение сустава $k$. По сути, это **взвешенное среднее** позиций вершин.

**Следствия:**
- Положения суставов зависят от формы тела ($\beta$)
- У полного человека суставы будут в немного других местах, чем у худого

In [ ]:
# Визуализация Joint Regressor
J_reg = smpl.J_regressor.cpu().numpy()  # (24, 6890)

print(f"Joint Regressor shape: {J_reg.shape}")
print(f"Non-zero entries: {np.count_nonzero(J_reg)} из {J_reg.size}")
print(f"Sparsity: {1 - np.count_nonzero(J_reg) / J_reg.size:.4%}")
print()

# Сколько вершин участвует в вычислении каждого сустава
for j in range(24):
    n_nonzero = np.count_nonzero(J_reg[j])
    print(f"  Joint {j:2d} ({SMPL_JOINT_NAMES[j]:>16s}): {n_nonzero:4d} vertices contribute")

In [ ]:
# Визуализация вершин, участвующих в вычислении выбранного сустава
JOINT_TO_VIS = 16

beta_zero = torch.zeros(1, 10, device=DEVICE)
theta_zero = torch.zeros(1, 72, device=DEVICE)
verts, joints = smpl(beta_zero, theta_zero)
v = verts[0].cpu().detach().numpy()
j = joints[0].cpu().detach().numpy()

# Веса регрессора для выбранного сустава
reg_weights = J_reg[JOINT_TO_VIS]  # (6890,)
contributing = reg_weights > 0

fig = go.Figure()
fig.add_trace(go.Mesh3d(
    x=v[:, 0], y=v[:, 1], z=v[:, 2],
    i=faces_np[:, 0], j=faces_np[:, 1], k=faces_np[:, 2],
    color="lightgray", opacity=0.3,
    name="Mesh",
))

# Вклад (вес) каждой вершины
v_contrib = v[contributing]
w_contrib = reg_weights[contributing]
fig.add_trace(go.Scatter3d(
    x=v_contrib[:, 0], y=v_contrib[:, 1], z=v_contrib[:, 2],
    mode="markers",
    marker=dict(size=4, color=w_contrib, colorscale="Hot", showscale=True,
                colorbar=dict(title="Weight", x=1.05)),
    name=f"J_reg vertices for {SMPL_JOINT_NAMES[JOINT_TO_VIS]}",
))

# Позиция самого сустава
fig.add_trace(go.Scatter3d(
    x=[j[JOINT_TO_VIS, 0]], y=[j[JOINT_TO_VIS, 1]], z=[j[JOINT_TO_VIS, 2]],
    mode="markers",
    marker=dict(size=5, color="blue", symbol="x"),
    name=f"Joint: {SMPL_JOINT_NAMES[JOINT_TO_VIS]}",
))

from utils.visualization import _SCENE_Y_UP
fig.update_layout(
    title=f"Joint Regressor: {SMPL_JOINT_NAMES[JOINT_TO_VIS]} (joint {JOINT_TO_VIS})",
    scene=_SCENE_Y_UP,
    width=900, height=600,
    legend=dict(x=0, y=1, xanchor="left"),
)
fig.show()

### Полная формула SMPL

Собирая всё вместе, forward pass модели SMPL:

$$M(\beta, \theta) = W\Big(T_P(\beta, \theta),\; J(\beta),\; \theta,\; \mathscr{W}\Big)$$

где:
1. **Shape blend shapes:** $T_s(\beta) = \bar{T} + B_S(\beta)$
2. **Joint regression:** $J(\beta) = \mathscr{J} \cdot T_s(\beta)$
3. **Pose blend shapes:** $T_P(\beta, \theta) = T_s(\beta) + B_P(\theta)$
4. **LBS:** $W(\cdot)$ — Linear Blend Skinning с весами $\mathscr{W}$

In [ ]:
# Визуализация полного скелета SMPL
from utils.visualization import plot_mesh_3d

beta_zero = torch.zeros(1, 10, device=DEVICE)
theta_zero = torch.zeros(1, 72, device=DEVICE)
verts, joints = smpl(beta_zero, theta_zero)

fig = plot_mesh_3d(
    verts[0], smpl.faces, color="lightblue", opacity=0.3,
    title="SMPL Mesh + Skeleton",
    joints=joints[0], skeleton=SMPL_SKELETON,
)
fig.show()

# Часть 2: Пайплайн восстановления SMPL по мультивью-изображениям

## 2.0 Обзор пайплайна

Задача: имея синхронизированное мультивью-видео и калибровки камер, восстановить параметры SMPL ($\beta$, $\theta$) для каждого кадра.

![pipeline](assets/pipeline.jpg)

## 2.1 Загрузка данных

In [ ]:
import cv2
from utils.camera import load_cameras

# Загрузка камер
cameras = load_cameras(str(DATA_DIR / "cameras.json"))
print(f"Loaded {len(cameras)} cameras: {list(cameras.keys())}")

for name, cam in cameras.items():
    print(f"  {name}: {cam.img_width}x{cam.img_height}, center={cam.center.round(3)}")

In [ ]:
# Визуализация расположения камер
from utils.visualization import plot_cameras_3d

fig = plot_cameras_3d(cameras)
fig.update_layout(title="Camera Setup")
fig.show()

In [ ]:
from PIL import Image as PILImage

# Загрузка изображений для одного кадра
FRAME_IDX = 0
frame_name = f"{FRAME_IDX:06d}"

images = {}
for cam_name in cameras:
    for ext in [".jpg", ".png"]:
        img_path = DATA_DIR / "images" / cam_name / f"{frame_name}{ext}"
        if img_path.exists():
            images[cam_name] = np.array(PILImage.open(img_path).convert("RGB"))
            break

print(f"Loaded images for frame {FRAME_IDX}: {len(images)} cameras")

# 4 ракурса (2x2)
show_cams = list(images.keys())[:4]
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, cam_name in zip(axes.flat, show_cams):
    ax.imshow(images[cam_name])
    ax.set_title(cam_name)
    ax.axis("off")
plt.suptitle(f"Frame {FRAME_IDX}: Multi-view images ({len(images)} cameras total)")
plt.tight_layout()
plt.show()

## 2.2 Детекция человека (YOLO)

In [ ]:
from utils.pose_estimation import PersonDetector

detector = PersonDetector(model_name="yolov8m.pt", device=DEVICE)

# Детекция для всех камер
all_boxes = {}
for cam_name, img in images.items():
    all_boxes[cam_name] = detector.detect(img, conf_threshold=0.45)

print(f"Detected persons in {len(all_boxes)} cameras:")
for cam_name, boxes in all_boxes.items():
    print(f"  {cam_name}: {len(boxes)} person(s)")

# Визуализация первых 4 камер
show_cams = list(images.keys())[:4]
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

for ax, cam_name in zip(axes.flat, show_cams):
    img = images[cam_name]
    boxes = all_boxes[cam_name]

    ax.imshow(img)
    for box in boxes:
        x1, y1, x2, y2, conf = box
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                              linewidth=2, edgecolor="red", facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, y1 - 5, f"{conf:.2f}", color="red", fontsize=10)
    ax.set_title(f"{cam_name}: {len(boxes)} persons")
    ax.axis("off")

plt.suptitle("YOLO Person Detection")
plt.tight_layout()
plt.show()

## 2.3 Оценка 2D-позы (ViTPose)

**ViTPose** (Xu et al., NeurIPS 2022) — модель оценки позы на основе Vision Transformer.
На вход подаётся кроп человека, на выход — 17 ключевых точек в формате COCO.

In [ ]:
from utils.pose_estimation import PoseEstimator, load_precomputed_keypoints
from utils.visualization import plot_skeleton_on_image, COCO_SKELETON
from utils.smpl_model import COCO_KEYPOINT_NAMES

estimator = PoseEstimator(device=DEVICE)

all_keypoints = {}
for cam_name, img in images.items():
    boxes = all_boxes[cam_name]
    if len(boxes) == 0:
        print(f"  {cam_name}: no person detected, skipping")
        continue
    # Оставляем самый уверенный детект
    best_idx = np.argmax(boxes[:, 4])
    best_box = boxes[best_idx:best_idx+1]

    kpts_list = estimator.estimate(img, best_box)
    if kpts_list:
        all_keypoints[cam_name] = kpts_list[0]  # (17, 3): x, y, conf
        print(f"  {cam_name}: detected {(kpts_list[0][:, 2] > 0.3).sum()}/17 keypoints")

print(f"\nTotal: keypoints for {len(all_keypoints)}/{len(images)} cameras")

In [ ]:
# Визуализация 2D ключевых точек на изображениях
show_cams = list(images.keys())[:4]
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

for ax, cam_name in zip(axes.flat, show_cams):
    img = images[cam_name]
    if cam_name in all_keypoints:
        plot_skeleton_on_image(
            img, all_keypoints[cam_name],
            skeleton=COCO_SKELETON,
            keypoint_names=COCO_KEYPOINT_NAMES,
            ax=ax,
        )
    else:
        ax.imshow(img)
    ax.set_title(cam_name)

plt.suptitle("ViTPose 2D Keypoints")
plt.tight_layout()
plt.show()

## 2.4 Триангуляция 3D-скелета

Имея 2D ключевые точки из нескольких камер + калибровки, можно восстановить 3D-координаты каждой точки методом **DLT** (Direct Linear Transform).

Для каждой точки с наблюдениями $(u_i, v_i)$ из камеры $i$ с проекционной матрицей $P_i = K_i [R_i | t_i]$:

$$u_i \cdot P_i^{(3)} - P_i^{(1)} = 0$$
$$v_i \cdot P_i^{(3)} - P_i^{(2)} = 0$$

Это даёт переопределённую однородную систему $A \mathbf{X} = 0$, решаемую через SVD.

In [ ]:
from utils.triangulation import triangulate_points

# Собираем 2D keypoints для триангуляции
kpts_for_triang = {}
for cam_name, kpts in all_keypoints.items():
    kpts_for_triang[cam_name] = np.array(kpts)  # (17, 3): x, y, conf

# Триангуляция
points_3d, reproj_errors, valid_mask = triangulate_points(
    kpts_for_triang, cameras, confidence_threshold=0.3
)

print(f"Triangulated {valid_mask.sum()}/17 keypoints")
print(f"Mean reprojection error: {reproj_errors[valid_mask].mean():.2f} px")
print()
for k in range(17):
    status = "OK" if valid_mask[k] else "FAILED"
    err = f"{reproj_errors[k]:.2f}" if valid_mask[k] else "N/A"
    print(f"  {COCO_KEYPOINT_NAMES[k]:>16s}: {status:>6s}  reproj={err:>6s} px  "
          f"pos={points_3d[k].round(3) if valid_mask[k] else 'N/A'}")

In [ ]:
# Визуализация 3D-скелета (в мировой системе координат, Y-down)
from utils.visualization import plot_skeleton_3d

fig = plot_skeleton_3d(
    points_3d, COCO_SKELETON,
    title="Triangulated 3D Skeleton (COCO)",
    joint_names=COCO_KEYPOINT_NAMES,
    y_up=False,  # мировая система: Y вниз
)

# Добавим камеры для контекста
fig = plot_cameras_3d(cameras, fig=fig, scale=0.2)
fig.show()

## 2.5 Оптимизация SMPL-параметров по скелету

Теперь подбираем параметры $\beta$, $\theta$ и трансляцию $t$, чтобы суставы SMPL были максимально близки к триангулированным 3D-точкам.

$$\min_{\beta, \theta, t} \sum_{k} \| J_k^{\text{SMPL}}(\beta, \theta, t) - J_k^{\text{triang}} \|^2 + \lambda_\beta \|\beta\|^2 + \lambda_\theta \|\theta\|^2$$

In [ ]:
from utils.optimization import fit_smpl_to_joints

# На реальной съемке ось Y направлена вниз, а у модели SMPL в каноническом виде
# ось Y направлена вверх, поэтому нужно помочь оптимизатору и сразу начать с 
# повернутой системой координат
init_thetas = torch.zeros(72)
init_thetas[0] = np.pi
# Чтобы стартовать рядом с целью
valid_joints = points_3d[valid_mask]
init_trans = torch.tensor(valid_joints.mean(axis=0), dtype=torch.float32)

# Оптимизация SMPL к COCO-скелету
betas_opt, thetas_opt, trans_opt, loss_hist = fit_smpl_to_joints(
    smpl,
    target_joints_3d=points_3d,
    n_iters=1000,
    lr=0.02,
    w_joint=1.0,
    w_beta=0.001,
    w_theta=0.0005,
    init_thetas=init_thetas,
    init_trans=init_trans,
    verbose=True,
)

print(f"\nOptimized betas: {betas_opt.detach().cpu().numpy().round(3)}")
print(f"Final loss: {loss_hist[-1]:.6f}")

In [ ]:
# График сходимости
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(loss_hist)
ax.set_xlabel("Iteration")
ax.set_ylabel("Loss")
ax.set_title("SMPL Fitting: Convergence")
ax.set_yscale("log")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Визуализация результата: SMPL mesh + скелет + триангулированные точки
from utils.visualization import plot_smpl_result

verts_opt, joints_opt = smpl(
    betas_opt.unsqueeze(0), thetas_opt.unsqueeze(0), trans_opt.unsqueeze(0)
)

fig = plot_mesh_3d(
    verts_opt[0], smpl.faces, color="lightblue", opacity=0.4,
    title="SMPL Fit to Skeleton",
    joints=joints_opt[0], skeleton=SMPL_SKELETON,
    y_down=True,
)

# Триангулированные точки
valid_pts = points_3d[valid_mask]
fig.add_trace(go.Scatter3d(
    x=valid_pts[:, 0], y=valid_pts[:, 1], z=valid_pts[:, 2],
    mode="markers",
    marker=dict(size=6, color="green", symbol="cross"),
    name="Triangulated (target)",
))

fig.show()

In [ ]:
# Репроекция SMPL суставов на изображения
from utils.visualization import plot_reprojection

j_opt = joints_opt[0].detach().cpu().numpy()

show_cams = list(images.keys())[:4]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, cam_name in zip(axes.flat, show_cams):
    cam = cameras[cam_name]
    proj_joints = cam.project(j_opt)
    detected_kpts = all_keypoints.get(cam_name)

    plot_reprojection(
        images[cam_name], proj_joints,
        detected_keypoints=detected_kpts,
        skeleton=SMPL_SKELETON,
        ax=ax,
        title=f"{cam_name}: SMPL reprojection",
    )

plt.suptitle("SMPL Reprojection vs Detected Keypoints")
plt.tight_layout()
plt.show()

In [ ]:
# Проекция полного SMPL-меша на реальные изображения
from utils.visualization import render_mesh_on_image

v_opt_np = verts_opt[0].detach().cpu().numpy()

show_cams = list(images.keys())[:4]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, cam_name in zip(axes.flat, show_cams):
    blended = render_mesh_on_image(
        images[cam_name], v_opt_np, faces_np,
        camera=cameras[cam_name],
        color=(100, 200, 255), alpha=0.7,
    )
    ax.imshow(blended)
    ax.set_title(cam_name)
    ax.axis("off")

plt.suptitle("SMPL Mesh Projection onto Images (after skeleton fit)")
plt.tight_layout()
plt.show()

## 2.6 Refinement по маскам (Silhouette IoU)

Идея в том, чтобы оптимизировать SMPL-параметры, пока SMPL-меш не начнет в точности 
повторять реальные маски человека

### Дифференцируемый рендер силуэта

Используем **PyTorch3D** `SoftSilhouetteShader` — дифференцируемую растеризацию треугольников. В отличие от обычной растеризации, грани рисуются с мягкими краями (soft rasterization), что делает рендер гладкой функцией от позиций вершин и позволяет считать градиенты.

### Компоненты лосса

**1. Silhouette IoU** — основной сигнал от масок (мультивью):

$$\text{IoU}(S, M) = \frac{\sum_{p} S_p \cdot M_p}{\sum_{p} (S_p + M_p - S_p \cdot M_p)}, \quad \mathscr{L}_{\text{mask}} = 1 - \frac{1}{|V|}\sum_{v \in V} \text{IoU}(S^{(v)}, M^{(v)})$$

где $V$ — подмножество камер, случайно выбираемое на каждой итерации (батчевание по видам).

**2. Joint loss** — удерживает скелет на месте, чтобы маски не «увели» суставы:

$$\mathscr{L}_{\text{joints}} = \frac{1}{|K|}\sum_{k \in K} \| J_k^{\text{SMPL}} - J_k^{\text{triang}} \|^2$$

**3. Регуляризация** — штрафует большие отклонения параметров (prior):

$$\mathscr{L}_{\text{reg}} = \lambda_\beta \|\beta\|^2 + \lambda_\theta \|\theta\|^2$$

**Полный лосс:**

$$\min_{\beta, \theta, t} \; w_{\text{mask}} \cdot \mathscr{L}_{\text{mask}} + w_j \cdot \mathscr{L}_{\text{joints}} + \lambda_\beta \|\beta\|^2 + \lambda_\theta \|\theta\|^2$$

In [ ]:
# Загрузка масок
from PIL import Image as PILImage

gt_masks = {}
for cam_name in cameras:
    for ext in [".jpg", ".png"]:
        mask_path = DATA_DIR / "masks" / cam_name / f"{frame_name}{ext}"
        if mask_path.exists():
            m = np.array(PILImage.open(mask_path).convert("L"))  # grayscale
            gt_masks[cam_name] = m
            break

print(f"Loaded masks for {len(gt_masks)}/{len(cameras)} cameras")

# Визуализация масок
from utils.visualization import render_mesh_on_image

v_before = verts_opt[0].detach().cpu().numpy()
show_cams = list(gt_masks.keys())[:4]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, cam_name in zip(axes.flat, show_cams):
    mask_rgb = np.stack([gt_masks[cam_name]] * 3, axis=-1)  # (H, W, 3)
    blended = render_mesh_on_image(
        mask_rgb, v_before, faces_np,
        camera=cameras[cam_name],
        color=(100, 200, 255), alpha=0.7,
    )
    ax.imshow(blended)
    ax.set_title(f"{cam_name}: mask + SMPL (before)")
    ax.axis("off")

plt.suptitle("GT Masks vs SMPL (skeleton fit only)")
plt.tight_layout()
plt.show()

In [ ]:
# Визуализация дифференцируемого рендера силуэта через PyTorch3D
from utils.optimization import render_silhouette
import torch.nn.functional as F

v_opt_t = verts_opt[0]  # tensor на DEVICE
render_h = 128

show_cams = list(gt_masks.keys())[:4]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for col, cam_name in enumerate(show_cams):
    cam = cameras[cam_name]
    orig_H, orig_W = gt_masks[cam_name].shape[:2]
    render_w = int(round(render_h * orig_W / orig_H))

    # Рендер силуэта
    with torch.no_grad():
        sil = render_silhouette(
            v_opt_t, smpl.faces, cam,
            orig_H, orig_W,
            render_h=render_h, render_w=render_w,
        )
    sil_np = sil.cpu().numpy()

    # GT-маска на том же разрешении
    m = gt_masks[cam_name].astype(np.float32)
    if m.max() > 1:
        m = m / 255.0
    m_t = torch.tensor(m).unsqueeze(0).unsqueeze(0)
    m_lr = F.interpolate(m_t, size=(render_h, render_w), mode="bilinear",
                         align_corners=False).squeeze().numpy()

    # Верхний ряд - рендеры силуэтов
    axes[0, col].imshow(sil_np, cmap="gray", vmin=0, vmax=1)
    axes[0, col].set_title(f"{cam_name}: rendered")
    axes[0, col].axis("off")

    # Нижний ряд: GT-маски
    axes[1, col].imshow(m_lr, cmap="gray", vmin=0, vmax=1)
    axes[1, col].set_title(f"{cam_name}: GT mask")
    axes[1, col].axis("off")

plt.suptitle(f"PyTorch3D Soft Silhouette ({render_h}px) vs GT Masks")
plt.tight_layout()
plt.show()

In [ ]:
from utils.optimization import fit_smpl_to_masks

# Оптимизация SMPL по маскам
betas_mask, thetas_mask, trans_mask, loss_hist_mask = fit_smpl_to_masks(
    smpl,
    masks=gt_masks,
    cameras=cameras,
    target_joints_3d=points_3d,
    n_iters=300,
    lr=0.005,
    w_mask=1.0,
    w_joint=0.1,
    w_beta=0.001,
    w_theta=0.0005,
    init_betas=betas_opt,
    init_thetas=thetas_opt,
    init_trans=trans_opt,
    render_h=128,
    n_views_per_iter=4,
    verbose=True,
)

# Результат
verts_mask, joints_mask = smpl(
    betas_mask.unsqueeze(0), thetas_mask.unsqueeze(0), trans_mask.unsqueeze(0)
)
print(f"\nFinal loss: {loss_hist_mask[-1]:.4f}")

In [ ]:
# График сходимости
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(loss_hist_mask)
ax.set_xlabel("Iteration")
ax.set_ylabel("Loss (1 - IoU)")
ax.set_title("Mask Fitting: Convergence")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Визуализация SMPL после фита по маскам
v_after = verts_mask[0].detach().cpu().numpy()

show_cams = list(gt_masks.keys())[:4]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, cam_name in zip(axes.flat, show_cams):
    mask_rgb = np.stack([gt_masks[cam_name]] * 3, axis=-1)
    blended = render_mesh_on_image(
        mask_rgb, v_after, faces_np,
        camera=cameras[cam_name],
        color=(100, 200, 255), alpha=0.5,
    )
    ax.imshow(blended)
    ax.set_title(f"{cam_name}: mask + SMPL (after)")
    ax.axis("off")

plt.suptitle("GT Masks vs SMPL (after mask refinement)")
plt.tight_layout()
plt.show()

# Также на реальных фото
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, cam_name in zip(axes.flat, show_cams):
    blended = render_mesh_on_image(
        images[cam_name], v_after, faces_np,
        camera=cameras[cam_name],
        color=(100, 200, 255), alpha=0.4,
    )
    ax.imshow(blended)
    ax.set_title(f"{cam_name}")
    ax.axis("off")

plt.suptitle("Final SMPL Mesh on Real Images (after mask refinement)")
plt.tight_layout()
plt.show()

In [ ]:
# Визуализация оптимизированного по маскам SMPL-меша
from utils.visualization import plot_smpl_result

fig = plot_mesh_3d(
    v_after, smpl.faces, color="lightblue", opacity=0.4,
    title="SMPL after optimization by masks",
    joints=joints_mask[0], skeleton=SMPL_SKELETON,
    y_down=True,
)

fig.show()

## 2.7 `✨Бонус` Трекинг по видео

У нас есть точный SMPL для одного кадра. Форма тела ($\beta$) не меняется между кадрами. Остаётся только дооптимизировать позу ($\theta$) и трансляцию ($t$) для каждого нового кадра.

**Подход:** последовательно идём от кадра к кадру. Для каждого нового кадра:
1. Инициализируем $\theta$, $t$ от предыдущего кадра (поза меняется плавно)
2. Запускаем несколько итераций оптимизации по маскам нового кадра
3. Сохраняем результат как инициализацию для следующего кадра

Это работает, потому что между соседними кадрами поза меняется незначительно, и 15-20 итераций достаточно для корректировки.

In [ ]:
import glob
from tqdm import tqdm

def load_masks_for_frame(frame_idx, cam_names, data_dir):
    masks = {}
    fname = f"{frame_idx:06d}"
    for cam_name in cam_names:
        for ext in [".jpg", ".png"]:
            p = data_dir / "masks" / cam_name / f"{fname}{ext}"
            if p.exists():
                masks[cam_name] = np.array(PILImage.open(p).convert("L"))
                break
    return masks

# Определяем количество доступных кадров
sample_cam = list(cameras.keys())[0]
mask_files = sorted(
    glob.glob(str(DATA_DIR / "masks" / sample_cam / "*.jpg"))
    + glob.glob(str(DATA_DIR / "masks" / sample_cam / "*.png"))
)#[:20]
n_frames = len(mask_files)
print(f"Available frames: {n_frames}")

# Фиксируем beta из уже оптимизированного SMPL-меша результата
betas_fixed = betas_mask.detach().clone()

# Последовательный трекинг
current_thetas = thetas_mask.detach().clone()
current_trans = trans_mask.detach().clone()
prev_thetas = current_thetas.clone()

all_verts = []  # все вершины на каждый кадр (для последующей анимации)

ITERS_PER_FRAME = 20

for frame_idx in tqdm(range(n_frames), desc="Tracking"):
    masks_f = load_masks_for_frame(frame_idx, cameras.keys(), DATA_DIR)

    if len(masks_f) < 4:
        v, _ = smpl(betas_fixed.unsqueeze(0), current_thetas.unsqueeze(0),
                     current_trans.unsqueeze(0))
        all_verts.append(v[0].detach().cpu().numpy())
        continue

    # Оптимизация кадра
    _, new_thetas, new_trans, _ = fit_smpl_to_masks(
        smpl,
        masks=masks_f,
        cameras=cameras,
        n_iters=ITERS_PER_FRAME,
        lr=0.01,
        w_mask=1.0,
        w_joint=0.0,
        w_beta=100.0,
        w_theta=0.0003,
        init_betas=betas_fixed,
        init_thetas=current_thetas,
        init_trans=current_trans,
        prev_thetas=prev_thetas,
        w_temporal=0.01,
        render_h=128,
        n_views_per_iter=4,
        verbose=False,
    )

    prev_thetas = current_thetas.clone()
    current_thetas = new_thetas.detach().clone()
    current_trans = new_trans.detach().clone()

    v, _ = smpl(betas_fixed.unsqueeze(0), current_thetas.unsqueeze(0),
                 current_trans.unsqueeze(0))
    all_verts.append(v[0].detach().cpu().numpy())

print(f"\nTracked {len(all_verts)} frames")

In [ ]:
# Анимация 3D SMPL-меша через plotly frames
from utils.visualization import _SCENE_Y_DOWN

f_np = faces_np

# Первый кадр
v0 = all_verts[0]
fig = go.Figure(
    data=[go.Mesh3d(
        x=v0[:, 0], y=v0[:, 1], z=v0[:, 2],
        i=f_np[:, 0], j=f_np[:, 1], k=f_np[:, 2],
        color="lightblue", opacity=0.7,
    )],
    layout=go.Layout(
        title="SMPL Tracking Animation",
        scene=_SCENE_Y_DOWN,
        width=800, height=600,
        updatemenus=[dict(
            type="buttons",
            showactive=False,
            y=0,
            x=0.5,
            xanchor="center",
            buttons=[
                dict(label="Play",
                     method="animate",
                     args=[None, dict(
                         frame=dict(duration=60, redraw=True),
                         fromcurrent=True,
                     )]),
                dict(label="Pause",
                     method="animate",
                     args=[[None], dict(
                         frame=dict(duration=0, redraw=False),
                         mode="immediate",
                     )]),
            ],
        )],
        sliders=[dict(
            active=0,
            steps=[dict(args=[[f"frame_{i}"],
                               dict(frame=dict(duration=60, redraw=True),
                                    mode="immediate")],
                         label=str(i),
                         method="animate")
                   for i in range(len(all_verts))],
            x=0.1, len=0.8,
            currentvalue=dict(prefix="Frame: "),
        )],
    ),
)

# Все кадры анимации
frames = []
for i, v in enumerate(all_verts):
    frames.append(go.Frame(
        data=[go.Mesh3d(
            x=v[:, 0], y=v[:, 1], z=v[:, 2],
            i=f_np[:, 0], j=f_np[:, 1], k=f_np[:, 2],
            color="lightblue", opacity=0.7,
        )],
        name=f"frame_{i}",
    ))
fig.frames = frames

fig.show()